# 7주차 실습 — 확산 모델링의 미래: GPT와 대형 모델 (8장)

**대상**: 전공 4학년 / **환경**: Google Colab (CPU, 추가 설치 불필요) / **소요**: 90~120분

---

## 학습 목표
강의자료 8장의 핵심 개념을 **직접 코드로 구현**하면서 수식과 그림이 실제로 무엇을 계산하는지 확인합니다.

| 문제 | 주제 | 강의자료 대응 |
|---|---|---|
| 1 | 생성적 사전학습 — 마스크 언어 모델링(MLM) | 그림 8-2 |
| 2 | 생성적 사전학습 — 마스크 이미지 모델링(MAE) | 그림 8-3 |
| 3 | 단일 모드 대조 학습 — SimCLR / NT-Xent | 그림 8-4 |
| 4 | 멀티모달 대조 학습 — CLIP & 제로샷 분류 | 그림 8-5 |
| 5 | GPT-1 — 인과적 self-attention과 목적함수 $L_1$, $L_2$ | 그림 8-9 |
| 6 | 병렬 학습 — 데이터 병렬 / 텐서 병렬 | 그림 8-6, 8-7 |
| 7 | InstructGPT / RLHF — 보상 모델과 KL 제약 목적함수 | 그림 8-15 |

## 진행 방법
1. **[환경 설정]** 셀을 먼저 실행합니다.
2. 각 문제의 `### TODO` 부분을 채워 함수를 완성합니다.
3. 바로 아래 **채점 셀**을 실행합니다. `[PASS]` 가 모두 출력되고 마지막에
   `문제 N 통과` 가 뜨면 정답입니다. 틀리면 `AssertionError` 가 납니다.
4. 문제는 앞에서 만든 함수를 뒤에서 다시 쓰는 경우가 있으니 **순서대로** 푸세요.

> **프레임워크 안내**: 자동 미분에 가려지는 수식을 직접 보기 위해 NumPy로 구현합니다.
> 각 문제 끝에 대응되는 PyTorch API를 적어 두었으니, 실제 연구 코드로 옮길 때 참고하세요.

In [1]:
# ============================================================
# [환경 설정] 이 셀을 가장 먼저 실행하세요. (설치 필요 없음 / CPU 수 초)
# ============================================================
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=120)

# ---------- 앞으로 계속 쓰는 도우미 함수 (직접 구현할 필요 없음) ----------
def softmax(x, axis=-1):
    """수치적으로 안정한 softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    """수치적으로 안정한 log-softmax."""
    x = x - np.max(x, axis=axis, keepdims=True)
    return x - np.log(np.sum(np.exp(x), axis=axis, keepdims=True))

def gelu(x):
    """GELU 활성화 함수 (tanh 근사). 그림 8-7의 Y = GeLU(XA) 에 사용."""
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def sigmoid(x):
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))

def logsigmoid(x):
    """log(sigmoid(x)) 를 안정적으로 계산."""
    return -np.logaddexp(0.0, -x)

def ok(msg):
    print("  [PASS]", msg)

print("설정 완료 / numpy:", np.__version__)

설정 완료 / numpy: 2.5.2


---
# 문제 1. 생성적 사전학습 ① — 마스크 언어 모델링 (MLM)

강의자료 **그림 8-2**: `All the [MASK] is best` 처럼 입력 토큰의 일부를 가린 뒤,
인코더가 **가려진 자리의 원래 토큰을 복원**하도록 학습하는 것이 생성적 사전학습(BERT 계열)입니다.

BERT의 표준 마스킹 규칙은 다음과 같습니다.

* 특수 토큰(`[CLS]`, `[SEP]`, `[PAD]`)은 **절대 선택하지 않는다.**
* 나머지 토큰을 각각 확률 `mask_prob`(=0.15)로 **선택**한다.
* 선택된 토큰 중
  * **80%** → `[MASK]` 토큰으로 치환
  * **10%** → 어휘집에서 뽑은 **임의의 토큰**으로 치환
  * **10%** → **원래 토큰 그대로** 둔다 (사전학습-미세조정 불일치 완화)
* 손실은 **선택된 위치에서만** 계산한다. → 선택되지 않은 위치의 정답 라벨은 `-100`(무시)으로 둔다.

$$\mathcal{L}_{\text{MLM}} = -\frac{1}{|M|}\sum_{i \in M} \log P(x_i \mid \tilde{x};\theta)$$

## 할 일
1. `mlm_mask_tokens` — 위 규칙대로 `(masked_inputs, labels)` 를 만든다.
2. `masked_cross_entropy` — `labels == ignore_index` 인 위치를 **제외**하고 평균 교차엔트로피를 구한다.

> **힌트**: 난수는 반드시 인자로 받은 `rng` 만 사용하세요. 채점이 시드에 의존합니다.
> 한 번 뽑은 균등난수 `r ~ U(0,1)` 하나로 `r < 0.8`, `0.8 <= r < 0.9`, `r >= 0.9` 를 나누면
> 80/10/10 을 깔끔하게 구현할 수 있습니다.

In [2]:
def mlm_mask_tokens(input_ids, vocab_size, mask_token_id, special_tokens_mask,
                    mask_prob=0.15, rng=None):
    """
    input_ids           : (B, T) int  원본 토큰 ID
    special_tokens_mask : (B, T) 0/1  1이면 특수 토큰(마스킹 금지)
    반환 masked_inputs  : (B, T) int  모델에 실제로 넣을 입력
         labels         : (B, T) int  선택된 위치는 원래 토큰, 나머지는 -100
    """
    rng = np.random.default_rng(0) if rng is None else rng
    input_ids = np.asarray(input_ids)
    special_tokens_mask = np.asarray(special_tokens_mask).astype(bool)
    prob = np.full(input_ids.shape, mask_prob, dtype=float)
    prob[special_tokens_mask] = 0.0
    selected = rng.random(input_ids.shape) < prob
    labels = np.full(input_ids.shape, -100, dtype=input_ids.dtype)
    labels[selected] = input_ids[selected]
    masked_inputs = input_ids.copy()
    replacement_random = rng.random(input_ids.shape)
    mask_positions = selected & (replacement_random < 0.8)
    random_positions = selected & (replacement_random >= 0.8) & (replacement_random < 0.9)
    masked_inputs[mask_positions] = mask_token_id
    random_tokens = rng.integers(0, vocab_size, size=input_ids.shape)
    masked_inputs[random_positions] = random_tokens[random_positions]
    return masked_inputs, labels


def masked_cross_entropy(logits, labels, ignore_index=-100):
    """logits : (B, T, V), labels : (B, T)"""
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    valid = labels != ignore_index
    if not np.any(valid):
        return 0.0
    logp = log_softmax(logits, axis=-1)
    safe_labels = np.where(valid, labels, 0)
    chosen = np.take_along_axis(logp, safe_labels[..., None], axis=-1)[..., 0]
    return float(-chosen[valid].mean())


In [3]:
# ===================== 문제 1 채점 =====================
rng = np.random.default_rng(42)
B, T, V = 64, 32, 500
MASK_ID = 4

input_ids = rng.integers(10, V, size=(B, T))
special = np.zeros((B, T), dtype=int)
special[:, 0] = 1        # [CLS]
special[:, -1] = 1       # [SEP]
input_ids[:, 0] = 1
input_ids[:, -1] = 2

test_rng = np.random.default_rng(2024)
mi, lb = mlm_mask_tokens(input_ids, V, MASK_ID, special, 0.15, test_rng)

assert mi.shape == input_ids.shape and lb.shape == input_ids.shape, "출력 shape가 다릅니다"
ok("출력 shape 정상")

assert np.all(lb[special.astype(bool)] == -100), "특수 토큰이 마스킹 대상으로 선택되었습니다"
assert np.all(mi[special.astype(bool)] == input_ids[special.astype(bool)]), "특수 토큰이 변형되었습니다"
ok("특수 토큰은 마스킹되지 않음")

sel = lb != -100
assert np.all(mi[~sel] == input_ids[~sel]), "선택되지 않은 위치가 변형되었습니다"
ok("선택되지 않은 위치는 원본 유지")

assert np.all(lb[sel] == input_ids[sel]), "labels 에 원래 토큰이 들어있지 않습니다"
ok("labels 가 원본 토큰을 담고 있음")

ratio = sel.sum() / special.size          # 특수 토큰 포함 전체 대비
eff = sel.sum() / (~special.astype(bool)).sum()
assert 0.12 < eff < 0.18, f"마스킹 비율이 15%에서 너무 벗어납니다 (={eff:.3f})"
ok(f"마스킹 비율 {eff*100:.1f}% (목표 15%)")

frac_mask_tok = (mi[sel] == MASK_ID).mean()
assert 0.74 < frac_mask_tok < 0.86, f"[MASK] 치환 비율이 80%가 아닙니다 (={frac_mask_tok:.3f})"
ok(f"선택된 토큰 중 [MASK] 치환 {frac_mask_tok*100:.1f}% (목표 80%)")

frac_same = (mi[sel] == input_ids[sel]).mean()
assert 0.05 < frac_same < 0.18, f"'원본 유지 10%' 규칙이 지켜지지 않았습니다 (={frac_same:.3f})"
ok(f"선택된 토큰 중 원본 유지 {frac_same*100:.1f}% (목표 ~10%)")

# --- 손실 함수 검증 ---
logits = rng.normal(size=(2, 5, 7))
labels = np.array([[3, -100, 1, -100, 0],
                   [-100, -100, 6, 2, -100]])
got = masked_cross_entropy(logits, labels)

ref = []
for b in range(2):
    for t in range(5):
        if labels[b, t] != -100:
            p = softmax(logits[b, t])
            ref.append(-np.log(p[labels[b, t]]))
ref = float(np.mean(ref))
assert abs(got - ref) < 1e-9, f"손실값 불일치: got={got}, expected={ref}"
ok(f"masked_cross_entropy = {got:.6f} (기준값과 일치)")

full = masked_cross_entropy(logits, np.zeros((2, 5), dtype=int))
assert abs(full - (-log_softmax(logits, -1)[..., 0].mean())) < 1e-9, "무시 위치가 없을 때 계산이 틀립니다"
ok("ignore_index 가 없는 경우도 정상")

print("\n✅ 문제 1 통과")

  [PASS] 출력 shape 정상
  [PASS] 특수 토큰은 마스킹되지 않음
  [PASS] 선택되지 않은 위치는 원본 유지
  [PASS] labels 가 원본 토큰을 담고 있음
  [PASS] 마스킹 비율 13.8% (목표 15%)
  [PASS] 선택된 토큰 중 [MASK] 치환 81.5% (목표 80%)
  [PASS] 선택된 토큰 중 원본 유지 9.4% (목표 ~10%)
  [PASS] masked_cross_entropy = 2.079951 (기준값과 일치)
  [PASS] ignore_index 가 없는 경우도 정상

✅ 문제 1 통과


> **PyTorch 대응**: `torch.nn.functional.cross_entropy(logits.view(-1, V), labels.view(-1), ignore_index=-100)`
> — HuggingFace `DataCollatorForLanguageModeling` 이 위와 똑같은 80/10/10 규칙을 사용합니다.

---
# 문제 2. 생성적 사전학습 ② — 마스크 이미지 모델링 (MAE)

강의자료 **그림 8-3**: 이미지를 패치로 자르고 **75%를 버린 뒤**, 남은 25%만 인코더에 넣고
디코더가 **버려진 패치를 복원**하게 하는 것이 MAE(Masked Auto-Encoder)입니다.
텍스트의 MLM과 완전히 같은 발상을 이미지로 옮긴 것이죠.

## 할 일
1. `patchify(imgs, p)` : `(B, C, H, W)` 이미지를 `(B, L, p*p*C)` 패치 시퀀스로 (`L = (H/p)*(W/p)`)
2. `random_masking(x, mask_ratio, rng)` : 각 샘플마다 **서로 다른** 위치를 남기도록 셔플하여
   - `x_keep` : `(B, len_keep, D)` 남긴 패치
   - `mask` : `(B, L)` — **버린 패치가 1**, 남긴 패치가 0
   - `ids_restore` : `(B, L)` — 원래 순서로 되돌리는 인덱스
3. `mae_loss(pred, target, mask)` : **버려진 패치에 대해서만** 패치별 MSE 평균

$$\mathcal{L}_{\text{MAE}}=\frac{1}{\sum m_i}\sum_i m_i \cdot \frac{1}{D}\lVert \hat{y}_i - y_i\rVert^2$$

> **핵심 트릭**: 난수 `noise`를 뽑아 `argsort` 하면 랜덤 순열이 됩니다.
> 앞쪽 `len_keep` 개를 남기고, `argsort(argsort(noise))` 가 곧 복원 인덱스입니다.

In [4]:
def patchify(imgs, p):
    """(B, C, H, W) -> (B, L, p*p*C)."""
    B, C, H, W = imgs.shape
    assert H % p == 0 and W % p == 0, "H, W 는 p 로 나누어떨어져야 합니다"
    h, w = H // p, W // p
    x = imgs.reshape(B, C, h, p, w, p)
    x = x.transpose(0, 2, 4, 3, 5, 1)
    return x.reshape(B, h * w, p * p * C)


def unpatchify(x, p, C, H, W):
    """patchify 의 역변환. (검증용으로 제공 — 수정하지 마세요)"""
    B = x.shape[0]
    h, w = H // p, W // p
    x = x.reshape(B, h, w, p, p, C)
    x = x.transpose(0, 5, 1, 3, 2, 4)
    return x.reshape(B, C, H, W)


def random_masking(x, mask_ratio, rng):
    """x : (B, L, D), mask에서 1은 버린 패치."""
    B, L, D = x.shape
    len_keep = int(L * (1 - mask_ratio))
    noise = rng.random((B, L))
    ids_shuffle = np.argsort(noise, axis=1)
    ids_restore = np.argsort(ids_shuffle, axis=1)
    ids_keep = ids_shuffle[:, :len_keep]
    x_keep = np.take_along_axis(x, ids_keep[:, :, None], axis=1)
    mask_shuffle = np.ones((B, L), dtype=float)
    mask_shuffle[:, :len_keep] = 0.0
    mask = np.take_along_axis(mask_shuffle, ids_restore, axis=1)
    return x_keep, mask, ids_restore


def mae_loss(pred, target, mask):
    """버려진 패치(mask=1)에 대해서만 패치별 MSE 평균."""
    patch_mse = ((np.asarray(pred) - np.asarray(target)) ** 2).mean(axis=-1)
    mask = np.asarray(mask)
    return float((patch_mse * mask).sum() / np.maximum(mask.sum(), 1.0))


In [5]:
# ===================== 문제 2 채점 =====================
rng = np.random.default_rng(7)
B, C, H, W, p = 4, 3, 16, 16, 4
imgs = rng.normal(size=(B, C, H, W))

x = patchify(imgs, p)
L, D = (H // p) * (W // p), p * p * C
assert x.shape == (B, L, D), f"patchify shape 오류: {x.shape} != {(B, L, D)}"
ok(f"patchify shape {x.shape}")

assert np.allclose(unpatchify(x, p, C, H, W), imgs), "patchify -> unpatchify 복원 실패 (축 순서를 확인하세요)"
ok("patchify ↔ unpatchify 왕복 복원 일치")

# 첫 번째 패치는 원본 좌상단 p×p 블록이어야 한다
assert np.allclose(x[0, 0].reshape(p, p, C).transpose(2, 0, 1), imgs[0, :, :p, :p]), \
    "첫 패치 내용이 좌상단 블록과 다릅니다"
ok("패치 배열 순서(행 우선) 정상")

# --- random_masking ---
mrng = np.random.default_rng(1234)
x_keep, mask, ids_restore = random_masking(x, 0.75, mrng)
len_keep = int(L * 0.25)
assert x_keep.shape == (B, len_keep, D), f"x_keep shape 오류: {x_keep.shape}"
assert mask.shape == (B, L) and ids_restore.shape == (B, L), "mask/ids_restore shape 오류"
ok(f"75% 마스킹 → 남은 패치 {len_keep}/{L}")

assert np.all(mask.sum(axis=1) == L - len_keep), "버린 패치 개수가 맞지 않습니다"
assert set(np.unique(mask)) <= {0.0, 1.0}, "mask 는 0/1 이어야 합니다"
ok("mask 개수/값 정상 (버린 곳=1)")

for b in range(B):
    assert sorted(ids_restore[b].tolist()) == list(range(L)), "ids_restore 가 순열이 아닙니다"
ok("ids_restore 가 유효한 순열")

# 남긴 패치는 실제로 mask==0 인 위치의 패치들과 집합이 같아야 한다
for b in range(B):
    kept_idx = np.where(mask[b] == 0)[0]
    A = np.sort(x[b][kept_idx], axis=0)
    Bv = np.sort(x_keep[b], axis=0)
    assert np.allclose(A, Bv), "x_keep 의 내용이 mask==0 위치와 일치하지 않습니다"
ok("x_keep 내용과 mask 위치 일치")

assert not np.array_equal(mask[0], mask[1]), "샘플마다 마스크가 달라야 합니다 (축을 확인하세요)"
ok("샘플별로 서로 다른 마스크")

# --- mae_loss ---
pred = rng.normal(size=(2, 4, 3))
target = rng.normal(size=(2, 4, 3))
m = np.array([[1., 0., 1., 0.], [0., 0., 1., 1.]])
got = mae_loss(pred, target, m)
ref_terms = [((pred[b, l] - target[b, l]) ** 2).mean()
             for b in range(2) for l in range(4) if m[b, l] == 1]
ref = float(np.mean(ref_terms))
assert abs(got - ref) < 1e-9, f"mae_loss 불일치: got={got}, expected={ref}"
ok(f"mae_loss = {got:.6f} (버려진 패치에서만 계산)")

perfect = mae_loss(target, target, m)
assert abs(perfect) < 1e-12, "완벽 복원 시 손실이 0이어야 합니다"
ok("완벽 복원 시 손실 0")

print("\n✅ 문제 2 통과")

  [PASS] patchify shape (4, 16, 48)
  [PASS] patchify ↔ unpatchify 왕복 복원 일치
  [PASS] 패치 배열 순서(행 우선) 정상
  [PASS] 75% 마스킹 → 남은 패치 4/16
  [PASS] mask 개수/값 정상 (버린 곳=1)
  [PASS] ids_restore 가 유효한 순열
  [PASS] x_keep 내용과 mask 위치 일치
  [PASS] 샘플별로 서로 다른 마스크
  [PASS] mae_loss = 2.233049 (버려진 패치에서만 계산)
  [PASS] 완벽 복원 시 손실 0

✅ 문제 2 통과


> **PyTorch 대응**: `torch.argsort`, `torch.gather(x, 1, idx.unsqueeze(-1).repeat(1,1,D))`.
> He et al., *Masked Autoencoders Are Scalable Vision Learners*, CVPR 2022 의 공식 구현과 동일한 방식입니다.

---
# 문제 3. 단일 모드 대조 학습 — SimCLR의 NT-Xent 손실

강의자료 **그림 8-4**: 같은 이미지 $x$ 에서 서로 다른 증강 $t, t'$ 로 만든 두 뷰 $\tilde{x}_i, \tilde{x}_j$ 의
표현 $z_i, z_j$ 는 **가깝게**(양성 쌍), 배치 내 다른 이미지의 표현과는 **멀게**(음성 쌍) 만듭니다.

배치 크기 $N$ 이면 $2N$ 개 표현을 이어 붙이고, 각 앵커 $i$ 에 대해

$$\ell_{i} = -\log\frac{\exp(\mathrm{sim}(z_i, z_{j(i)})/\tau)}{\sum_{k=1}^{2N}\mathbf{1}_{[k\neq i]}\exp(\mathrm{sim}(z_i, z_k)/\tau)},\qquad
\mathrm{sim}(u,v)=\frac{u^\top v}{\lVert u\rVert\lVert v\rVert}$$

최종 손실은 $2N$ 개 $\ell_i$ 의 평균입니다.

## 할 일
1. `l2_normalize(x)` — 마지막 축 기준 L2 정규화
2. `nt_xent_loss(z_i, z_j, tau)` — 위 식을 **벡터화**하여 구현
   * **자기 자신($k=i$)은 분모에서 반드시 제외**해야 합니다. (해당 위치를 `-inf` 로 만들면 됩니다)
   * 앵커 $i \in [0, N)$ 의 양성은 $i+N$, 앵커 $i \in [N, 2N)$ 의 양성은 $i-N$ 입니다.

> `-inf` 를 쓰면 `softmax` 에서 자연스럽게 0이 되므로 `log_softmax` 후 정답 인덱스만 뽑으면 됩니다.

In [6]:
def l2_normalize(x, axis=-1, eps=1e-12):
    x = np.asarray(x)
    norm = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(norm, eps)


def nt_xent_loss(z_i, z_j, tau=0.5):
    """z_i, z_j : (N, d) 같은 인덱스끼리 양성 쌍."""
    N = z_i.shape[0]
    z = l2_normalize(np.concatenate([z_i, z_j], axis=0))
    sim = (z @ z.T) / tau
    np.fill_diagonal(sim, -np.inf)
    targets = np.concatenate([np.arange(N, 2 * N), np.arange(0, N)])
    return float(-log_softmax(sim, axis=-1)[np.arange(2 * N), targets].mean())


In [7]:
# ===================== 문제 3 채점 =====================
rng = np.random.default_rng(11)

v = rng.normal(size=(3, 5))
vn = l2_normalize(v)
assert np.allclose(np.linalg.norm(vn, axis=-1), 1.0), "정규화 후 노름이 1이 아닙니다"
assert np.allclose(vn * np.linalg.norm(v, axis=-1, keepdims=True), v), "방향이 바뀌었습니다"
ok("l2_normalize 정상")

# --- 이중 루프 기준 구현과 비교 ---
def nt_xent_reference(z_i, z_j, tau):
    N = z_i.shape[0]
    z = l2_normalize(np.concatenate([z_i, z_j], 0))
    total = 0.0
    for i in range(2 * N):
        pos = i + N if i < N else i - N
        num = np.exp(np.dot(z[i], z[pos]) / tau)
        den = sum(np.exp(np.dot(z[i], z[k]) / tau) for k in range(2 * N) if k != i)
        total += -np.log(num / den)
    return total / (2 * N)

for N, d, tau in [(4, 8, 0.5), (6, 3, 0.1), (2, 16, 1.0)]:
    a = rng.normal(size=(N, d)); b = rng.normal(size=(N, d))
    got, ref = nt_xent_loss(a, b, tau), nt_xent_reference(a, b, tau)
    assert abs(got - ref) < 1e-8, f"N={N}, tau={tau} 에서 불일치: {got} vs {ref}"
ok("이중 루프 기준 구현과 수치 일치 (3가지 설정)")

# 양성 쌍이 완전히 일치하면 손실이 이론적 하한에 가까워야 한다
N, d = 8, 16
base = l2_normalize(rng.normal(size=(N, d)))
loss_perfect = nt_xent_loss(base, base.copy(), 0.05)
loss_random = nt_xent_loss(rng.normal(size=(N, d)), rng.normal(size=(N, d)), 0.05)
assert loss_perfect < loss_random, "양성 쌍이 동일할 때 손실이 더 작아야 합니다"
ok(f"동일 뷰 손실 {loss_perfect:.4f} < 무작위 뷰 손실 {loss_random:.4f}")

# 자기 자신을 분모에서 빼지 않으면 값이 달라진다 → 흔한 실수 검출
z = l2_normalize(np.concatenate([base, base.copy()], 0))
sim_wrong = (z @ z.T) / 0.05
tgt = np.concatenate([np.arange(N, 2 * N), np.arange(0, N)])
wrong = float(-log_softmax(sim_wrong, -1)[np.arange(2 * N), tgt].mean())
assert abs(loss_perfect - wrong) > 1e-3, "자기 자신(대각 성분)을 분모에서 제외하지 않았습니다"
ok("자기 유사도 제외 확인")

print(f"\n손실값 예시: {nt_xent_loss(rng.normal(size=(4,8)), rng.normal(size=(4,8)), 0.5):.4f}")
print("\n✅ 문제 3 통과")

  [PASS] l2_normalize 정상
  [PASS] 이중 루프 기준 구현과 수치 일치 (3가지 설정)
  [PASS] 동일 뷰 손실 0.0000 < 무작위 뷰 손실 14.5153
  [PASS] 자기 유사도 제외 확인

손실값 예시: 1.9417

✅ 문제 3 통과


> **PyTorch 대응**: `F.normalize(z, dim=-1)`, `sim.masked_fill_(torch.eye(2*N, dtype=bool), -float("inf"))`,
> `F.cross_entropy(sim, targets)`. Chen et al., *SimCLR*, ICML 2020.

---
# 문제 4. 멀티모달 대조 학습 — CLIP과 제로샷 분류

강의자료 **그림 8-5**: 이미지 인코더와 텍스트 인코더가 각각 $z_x, z_y$ 를 만들고,
$N\times N$ 유사도 행렬에서 **대각 성분(올바른 이미지-텍스트 쌍)의 상호정보를 최대화**,
나머지 $N^2-N$ 개(잘못된 쌍)를 최소화합니다.

$$\text{logits} = \exp(t)\cdot \hat{z}_x \hat{z}_y^\top,\qquad
\mathcal{L}=\tfrac{1}{2}\big[\mathrm{CE}(\text{logits}, \mathbf{I}) + \mathrm{CE}(\text{logits}^\top, \mathbf{I})\big]$$

여기서 정답 라벨은 `[0, 1, ..., N-1]` 이고, `logit_scale`$=\exp(t)$ 는 학습되는 온도의 역수입니다.

## 할 일
1. `clip_logits(img_emb, txt_emb, logit_scale)` → `(N, N)` 유사도 로짓 (행=이미지, 열=텍스트)
2. `clip_loss(...)` → 위의 **대칭** 교차엔트로피
3. `zero_shot_classify(img_emb, class_text_emb)` → 각 이미지에 대해 코사인 유사도가 최대인 클래스 인덱스

> 대조 학습으로 얻은 정렬(alignment) 덕분에, **한 번도 라벨을 안 본 분류기**가 만들어집니다.
> 이것이 강의자료에서 말한 "멀티모달 대조 사전 학습으로의 일반화"의 실체입니다.

In [8]:
def clip_logits(img_emb, txt_emb, logit_scale=100.0):
    """img_emb (N, d), txt_emb (N, d) -> (N, N) 로짓."""
    return logit_scale * (l2_normalize(img_emb) @ l2_normalize(txt_emb).T)


def clip_loss(img_emb, txt_emb, logit_scale=100.0):
    """이미지->텍스트, 텍스트->이미지 대칭 교차엔트로피."""
    logits = clip_logits(img_emb, txt_emb, logit_scale)
    labels = np.arange(logits.shape[0])
    ce_i2t = -log_softmax(logits, axis=-1)[labels, labels].mean()
    ce_t2i = -log_softmax(logits.T, axis=-1)[labels, labels].mean()
    return float((ce_i2t + ce_t2i) / 2.0)


def zero_shot_classify(img_emb, class_text_emb):
    """각 이미지와 가장 유사한 클래스 텍스트의 인덱스를 반환."""
    similarity = l2_normalize(img_emb) @ l2_normalize(class_text_emb).T
    return np.argmax(similarity, axis=1)


In [9]:
# ===================== 문제 4 채점 =====================
rng = np.random.default_rng(21)
N, d = 6, 12
I = rng.normal(size=(N, d)); Tt = rng.normal(size=(N, d))

lg = clip_logits(I, Tt, 100.0)
assert lg.shape == (N, N), "로짓 shape 오류"
assert np.all(np.abs(lg) <= 100.0 + 1e-6), "정규화를 하지 않은 것 같습니다 (|logit| <= scale 이어야 함)"
assert np.allclose(lg / 100.0, l2_normalize(I) @ l2_normalize(Tt).T), "코사인 유사도 계산이 다릅니다"
ok("clip_logits 정상 (코사인 × scale)")

# 스케일만 바꿔도 상대 순서는 유지
assert np.allclose(clip_logits(I, Tt, 1.0) * 50, clip_logits(I, Tt, 50.0)), "logit_scale 반영 오류"
ok("logit_scale 선형 반영")

# 대칭 손실 기준값
lab = np.arange(N)
ref = float((-log_softmax(lg, -1)[lab, lab].mean() + -log_softmax(lg.T, -1)[lab, lab].mean()) / 2)
got = clip_loss(I, Tt, 100.0)
assert abs(got - ref) < 1e-9, f"clip_loss 불일치: {got} vs {ref}"
ok(f"clip_loss = {got:.6f} (대칭 CE 일치)")

assert abs(clip_loss(I, Tt, 100.0) - clip_loss(Tt, I, 100.0)) < 1e-9, "손실이 대칭이 아닙니다"
ok("이미지↔텍스트 방향 대칭성 확인")

# 완벽히 정렬된 임베딩이면 손실이 거의 0
aligned = l2_normalize(rng.normal(size=(N, d)))
assert clip_loss(aligned, aligned.copy(), 100.0) < 1e-6, "완벽 정렬 시 손실이 0에 가까워야 합니다"
ok("완벽 정렬 시 손실 ≈ 0")

# --- 제로샷 분류 ---
C, M = 4, 40
proto = l2_normalize(rng.normal(size=(C, d)))          # 클래스별 텍스트 프롬프트 임베딩
y_true = rng.integers(0, C, size=M)
imgs = l2_normalize(proto[y_true] + 0.15 * rng.normal(size=(M, d)))  # 노이즈 섞인 이미지 임베딩

pred = zero_shot_classify(imgs, proto)
assert pred.shape == (M,), "예측 shape 오류"
acc = (pred == y_true).mean()
assert acc > 0.9, f"제로샷 정확도가 너무 낮습니다 ({acc:.2f})"
ok(f"제로샷 분류 정확도 {acc*100:.1f}% (라벨 학습 없이)")

# 스케일에 불변해야 함 (코사인 유사도이므로)
assert np.array_equal(zero_shot_classify(imgs * 7.3, proto), pred), "임베딩 크기에 영향을 받습니다 (정규화 확인)"
ok("임베딩 크기(norm)에 불변")

print("\n✅ 문제 4 통과")

  [PASS] clip_logits 정상 (코사인 × scale)
  [PASS] logit_scale 선형 반영
  [PASS] clip_loss = 23.425893 (대칭 CE 일치)
  [PASS] 이미지↔텍스트 방향 대칭성 확인
  [PASS] 완벽 정렬 시 손실 ≈ 0
  [PASS] 제로샷 분류 정확도 100.0% (라벨 학습 없이)
  [PASS] 임베딩 크기(norm)에 불변

✅ 문제 4 통과


> **PyTorch 대응**: OpenAI CLIP 공식 구현의 `logit_scale.exp() * image_features @ text_features.t()` 와 동일.
> Radford et al., *Learning Transferable Visual Models From Natural Language Supervision*, 2021.

---
# 문제 5. GPT-1 — 인과적 self-attention과 목적함수 $L_1$, $L_2$

강의자료의 GPT-1 목적함수:

$$L_1 = \sum_i \log P(u_i \mid u_{i-1},\dots,u_{i-k};\theta) \qquad\text{(사전학습, 언어모델링)}$$
$$L_2 = \sum_{(u,y)} \log P(y \mid u_1,\dots,u_m) \qquad\text{(미세조정, 지도학습)}$$
$$L_3 = L_2 + \lambda L_1 \qquad\text{(그림 8-9: 사전학습 + 미세조정 결합)}$$

$L_1$ 이 성립하려면 토큰 $i$ 를 예측할 때 **미래 토큰을 절대 보면 안 됩니다.**
이를 강제하는 것이 **인과 마스크(causal mask)** 입니다.

> 논문의 $L_1, L_2$ 는 **최대화** 대상이므로, 코드에서는 부호를 뒤집어
> $\mathcal{L} = -L_2 - \lambda L_1 = \mathrm{CE}_{\text{cls}} + \lambda\,\mathrm{CE}_{\text{lm}}$ 를 **최소화**합니다.

## 할 일
1. `causal_mask(T)` : `(T, T)` bool. `mask[i, j] = True` 이면 위치 $i$ 가 위치 $j$ 를 **볼 수 있음**
2. `causal_self_attention(Q, K, V)` : $\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + \text{mask}\right)V$
3. `lm_loss_L1(logits, tokens)` : 다음 토큰 예측 교차엔트로피 (**한 칸 shift** 필요)
4. `gpt1_objective(cls_logits, y, lm_logits, tokens, lam)` : $\mathrm{CE}_{\text{cls}} + \lambda\,\mathrm{CE}_{\text{lm}}$

In [10]:
def causal_mask(T):
    """(T, T) bool. True = 볼 수 있음 (하삼각, 대각 포함)"""
    return np.tril(np.ones((T, T), dtype=bool))


def causal_self_attention(Q, K, V):
    """Q, K, V : (B, T, d)"""
    B, T, d = Q.shape
    scores = np.einsum("btd,bsd->bts", Q, K) / np.sqrt(d)
    scores = np.where(causal_mask(T)[None, :, :], scores, -np.inf)
    attn = softmax(scores, axis=-1)
    out = np.einsum("bts,bsd->btd", attn, V)
    return out, attn


def lm_loss_L1(logits, tokens):
    """다음 토큰 예측 교차엔트로피."""
    logits = np.asarray(logits)
    tokens = np.asarray(tokens)
    logp = log_softmax(logits[:, :-1, :], axis=-1)
    targets = tokens[:, 1:]
    chosen = np.take_along_axis(logp, targets[..., None], axis=-1)[..., 0]
    return float(-chosen.mean())


def gpt1_objective(cls_logits, y, lm_logits, tokens, lam=0.5):
    """total = ce_cls + lam * ce_lm"""
    cls_logp = log_softmax(cls_logits, axis=-1)
    y = np.asarray(y)
    ce_cls = float(-cls_logp[np.arange(len(y)), y].mean())
    ce_lm = lm_loss_L1(lm_logits, tokens)
    return float(ce_cls + lam * ce_lm), ce_cls, ce_lm


In [11]:
# ===================== 문제 5 채점 =====================
rng = np.random.default_rng(33)

m = causal_mask(5)
assert m.shape == (5, 5) and m.dtype == bool, "causal_mask shape/dtype 오류"
assert np.array_equal(m, np.tril(np.ones((5, 5), bool))), "하삼각(대각 포함) 형태가 아닙니다"
assert m[0].sum() == 1 and m[4].sum() == 5, "첫 토큰은 자기 자신만, 마지막 토큰은 전부 봐야 합니다"
ok("causal_mask 정상")

B, T, d = 3, 6, 8
Q = rng.normal(size=(B, T, d)); K = rng.normal(size=(B, T, d)); V = rng.normal(size=(B, T, d))
out, attn = causal_self_attention(Q, K, V)

assert out.shape == (B, T, d) and attn.shape == (B, T, T), "출력 shape 오류"
ok(f"attention 출력 shape {out.shape}, {attn.shape}")

assert np.allclose(attn.sum(-1), 1.0), "attention 가중치의 행 합이 1이 아닙니다"
ok("attention 가중치 합 = 1")

upper = np.triu(np.ones((T, T), bool), k=1)
assert np.allclose(attn[:, upper], 0.0), "미래 토큰에 0이 아닌 attention 이 걸려 있습니다"
ok("미래 토큰 attention = 0 (인과성 확보)")

# 첫 토큰의 출력은 V[0] 과 같아야 한다
assert np.allclose(out[:, 0, :], V[:, 0, :]), "첫 토큰 출력은 V의 첫 벡터와 같아야 합니다"
ok("첫 토큰 출력 = V[0]")

# 인과성 실검증: 미래 토큰을 바꿔도 앞쪽 출력은 불변이어야 한다
K2, V2 = K.copy(), V.copy()
K2[:, -1] += 100.0; V2[:, -1] += 100.0
out2, _ = causal_self_attention(Q, K2, V2)
assert np.allclose(out[:, :-1], out2[:, :-1]), "미래 토큰 변경이 과거 출력에 영향을 줍니다"
ok("미래 토큰을 바꿔도 과거 출력 불변")

# --- L1 ---
V_ = 9
logits = rng.normal(size=(2, 4, V_))
tokens = rng.integers(0, V_, size=(2, 4))
got = lm_loss_L1(logits, tokens)
ref = float(np.mean([-log_softmax(logits[b, t])[tokens[b, t + 1]]
                     for b in range(2) for t in range(3)]))
assert abs(got - ref) < 1e-9, f"lm_loss_L1 불일치: {got} vs {ref}"
ok(f"lm_loss_L1 = {got:.6f} (shift 정확)")

# shift 를 안 하면(자기 자신 예측) 값이 달라야 함
same = float(np.mean([-log_softmax(logits[b, t])[tokens[b, t]] for b in range(2) for t in range(4)]))
assert abs(got - same) > 1e-6, "다음 토큰이 아니라 현재 토큰을 예측하고 있습니다"
ok("현재 토큰이 아닌 '다음' 토큰을 예측함")

# --- L3 = L2 + lam*L1 ---
cls_logits = rng.normal(size=(2, 3))
y = np.array([1, 2])
tot, ce_cls, ce_lm = gpt1_objective(cls_logits, y, logits, tokens, lam=0.5)
ref_cls = float(-np.mean([log_softmax(cls_logits[b])[y[b]] for b in range(2)]))
assert abs(ce_cls - ref_cls) < 1e-9, "분류 교차엔트로피가 틀립니다"
assert abs(ce_lm - got) < 1e-9, "언어모델 손실이 lm_loss_L1 과 다릅니다"
assert abs(tot - (ref_cls + 0.5 * got)) < 1e-9, "total = ce_cls + lam*ce_lm 이 아닙니다"
ok(f"L3 결합 손실 정상 (total={tot:.4f} = {ce_cls:.4f} + 0.5×{ce_lm:.4f})")

t0, _, _ = gpt1_objective(cls_logits, y, logits, tokens, lam=0.0)
assert abs(t0 - ref_cls) < 1e-9, "lam=0 이면 분류 손실만 남아야 합니다"
ok("lam=0 일 때 미세조정 손실만 사용")

print("\n✅ 문제 5 통과")

  [PASS] causal_mask 정상
  [PASS] attention 출력 shape (3, 6, 8), (3, 6, 6)
  [PASS] attention 가중치 합 = 1
  [PASS] 미래 토큰 attention = 0 (인과성 확보)
  [PASS] 첫 토큰 출력 = V[0]
  [PASS] 미래 토큰을 바꿔도 과거 출력 불변
  [PASS] lm_loss_L1 = 2.685095 (shift 정확)
  [PASS] 현재 토큰이 아닌 '다음' 토큰을 예측함
  [PASS] L3 결합 손실 정상 (total=2.5103 = 1.1677 + 0.5×2.6851)
  [PASS] lam=0 일 때 미세조정 손실만 사용

✅ 문제 5 통과


> **PyTorch 대응**: `torch.tril`, `scores.masked_fill(mask == 0, float("-inf"))`,
> 또는 `F.scaled_dot_product_attention(q, k, v, is_causal=True)`.
> Radford et al., *Improving Language Understanding by Generative Pre-Training*, 2018.

---
# 문제 6. 병렬 학습 — 데이터 병렬 vs 텐서 병렬

## (a) 데이터 병렬 (그림 8-6)
모든 GPU가 **모델 전체 사본**을 갖고, **배치를 쪼개** 각자 기울기를 계산한 뒤 **평균(all-reduce)** 합니다.
배치를 균등하게 나누면 **평균 기울기 = 전체 배치 기울기** 가 정확히 성립합니다. 이걸 직접 확인합니다.

선형 모델 $\hat{Y}=XW$, 손실 $\mathcal{L}=\frac{1}{ND}\lVert XW-Y\rVert_F^2$ 의 해석적 기울기는

$$\nabla_W \mathcal{L}=\frac{2}{ND}X^\top(XW-Y)$$

## (b) 텐서 병렬 (그림 8-7(a), Megatron-LM)
MLP $Z = \mathrm{GeLU}(XA)B$ 를 GPU $n$ 개로 쪼갭니다.

* $A = [A_1, A_2, \dots]$ — **열 방향(column) 분할** → GeLU 는 원소별 함수라 통신 없이 각자 계산 가능
* $B = [B_1; B_2; \dots]$ — **행 방향(row) 분할**
* 각 GPU가 $\mathrm{GeLU}(XA_i)B_i$ 를 계산한 뒤 **부분합을 all-reduce(덧셈)** → 단일 GPU 결과와 **정확히 동일**

> 왜 $A$ 는 열, $B$ 는 행으로 쪼갤까요? $A$ 를 행으로 쪼개면 GeLU 이전에 all-reduce 가 필요해집니다.
> 이 조합이라야 **MLP 블록당 통신이 딱 한 번**으로 끝납니다.

## 할 일
1. `mse_grad(X, Y, W)` — 해석적 기울기
2. `data_parallel_grad(X, Y, W, num_workers)` — 배치를 균등 분할 → 각자 `mse_grad` → 평균
3. `tensor_parallel_mlp(X, A, B, num_devices)` — 열/행 분할 후 부분합 합산

In [12]:
def mlp_single_device(X, A, B):
    """단일 GPU 기준 구현 (그림 8-7(a)). 수정하지 마세요."""
    return gelu(X @ A) @ B


def mse_grad(X, Y, W):
    """L = mean((XW - Y)^2) 의 dL/dW."""
    N, D = Y.shape
    return (2.0 / (N * D)) * X.T @ (X @ W - Y)


def data_parallel_grad(X, Y, W, num_workers):
    """배치를 num_workers 개로 균등 분할 -> 각자 기울기 -> 평균."""
    N = X.shape[0]
    assert N % num_workers == 0, "이 실습에서는 배치를 균등 분할합니다"
    x_parts = np.split(X, num_workers, axis=0)
    y_parts = np.split(Y, num_workers, axis=0)
    return np.mean([mse_grad(x_part, y_part, W) for x_part, y_part in zip(x_parts, y_parts)], axis=0)


def tensor_parallel_mlp(X, A, B, num_devices):
    """A 열 분할, B 행 분할 후 부분합을 all-reduce."""
    assert A.shape[1] % num_devices == 0, "hidden 차원이 장치 수로 나누어떨어져야 합니다"
    A_parts = np.split(A, num_devices, axis=1)
    B_parts = np.split(B, num_devices, axis=0)
    partials = [gelu(X @ A_part) @ B_part for A_part, B_part in zip(A_parts, B_parts)]
    return np.sum(partials, axis=0)


In [13]:
# ===================== 문제 6 채점 =====================
rng = np.random.default_rng(55)

# --- (a) 데이터 병렬 ---
N, din, dout = 64, 5, 3
X = rng.normal(size=(N, din)); Y = rng.normal(size=(N, dout)); W = rng.normal(size=(din, dout))

g = mse_grad(X, Y, W)
assert g.shape == W.shape, "기울기 shape 이 W 와 같아야 합니다"

# 수치 미분으로 해석적 기울기 검증
def loss_fn(W_):
    return float(np.mean((X @ W_ - Y) ** 2))
num_g = np.zeros_like(W)
eps = 1e-6
for i in range(din):
    for j in range(dout):
        Wp = W.copy(); Wp[i, j] += eps
        Wm = W.copy(); Wm[i, j] -= eps
        num_g[i, j] = (loss_fn(Wp) - loss_fn(Wm)) / (2 * eps)
assert np.allclose(g, num_g, atol=1e-6), "해석적 기울기가 수치 미분과 다릅니다"
ok("mse_grad 가 수치 미분과 일치")

for k in [1, 2, 4, 8]:
    gd = data_parallel_grad(X, Y, W, k)
    assert np.allclose(gd, g, atol=1e-12), f"GPU {k}개 데이터 병렬 결과가 전체 배치와 다릅니다"
ok("데이터 병렬 평균 기울기 = 전체 배치 기울기 (1/2/4/8 GPU)")

# 흔한 오해 검증: 평균이 아니라 합을 쓰면 틀린다
gs = data_parallel_grad(X, Y, W, 4)
assert not np.allclose(gs * 4, g), "기울기를 합산(sum)한 것 같습니다. all-reduce 후 평균이어야 합니다"
ok("합산이 아닌 평균으로 집계함")

# --- (b) 텐서 병렬 ---
Bsz, din2, hid, dout2 = 16, 12, 32, 8
Xt = rng.normal(size=(Bsz, din2))
A = rng.normal(size=(din2, hid)) * 0.3
Bm = rng.normal(size=(hid, dout2)) * 0.3

ref = mlp_single_device(Xt, A, Bm)
for nd in [1, 2, 4, 8]:
    out = tensor_parallel_mlp(Xt, A, Bm, nd)
    assert out.shape == ref.shape, f"장치 {nd}개 출력 shape 오류: {out.shape}"
    assert np.allclose(out, ref, atol=1e-10), f"장치 {nd}개 텐서 병렬 결과가 단일 장치와 다릅니다"
ok("텐서 병렬 결과 = 단일 장치 결과 (1/2/4/8 장치)")

# A 를 '행 방향'으로 쪼개면? → GeLU 이전에 all-reduce 를 해야만 맞는다
A1, A2 = np.split(A, 2, axis=0)                 # (din/2, hid) 씩
u1, u2 = Xt[:, :din2 // 2] @ A1, Xt[:, din2 // 2:] @ A2   # 각 장치의 부분합

naive = gelu(u1) @ Bm + gelu(u2) @ Bm           # 통신 없이 GeLU 를 먼저 적용 → 틀림
assert not np.allclose(naive, ref), "GeLU 는 비선형이므로 부분합에 먼저 적용하면 안 됩니다"

fixed = gelu(u1 + u2) @ Bm                      # GeLU 이전에 all-reduce → 맞음(통신 2회)
assert np.allclose(fixed, ref, atol=1e-10), "GeLU 이전에 all-reduce 하면 결과가 같아야 합니다"
ok("행 분할은 GeLU 이전 all-reduce 가 필요 → 열/행 조합이 통신 1회로 유리함을 확인")

print(f"\n단일 장치 출력 합계 : {ref.sum():.6f}")
print(f"4-way 텐서 병렬 합계: {tensor_parallel_mlp(Xt, A, Bm, 4).sum():.6f}")
print("\n✅ 문제 6 통과")

  [PASS] mse_grad 가 수치 미분과 일치
  [PASS] 데이터 병렬 평균 기울기 = 전체 배치 기울기 (1/2/4/8 GPU)
  [PASS] 합산이 아닌 평균으로 집계함
  [PASS] 텐서 병렬 결과 = 단일 장치 결과 (1/2/4/8 장치)
  [PASS] 행 분할은 GeLU 이전 all-reduce 가 필요 → 열/행 조합이 통신 1회로 유리함을 확인

단일 장치 출력 합계 : -28.766922
4-way 텐서 병렬 합계: -28.766922

✅ 문제 6 통과


> **실제 대응**: `torch.nn.parallel.DistributedDataParallel` 이 (a)의 all-reduce 평균을 자동으로 해 줍니다.
> (b)는 Megatron-LM 의 `ColumnParallelLinear` / `RowParallelLinear` 조합 그 자체입니다.
> Shoeybi et al., *Megatron-LM*, arXiv:1909.08053.

---
# 문제 7. InstructGPT / ChatGPT — 보상 모델과 KL 제약 목적함수

강의자료 **그림 8-15**의 3단계 중 2, 3단계를 구현합니다.

**Step 2 — 보상 모델 학습**: 라벨러가 출력을 순위 매긴 데이터로,
선호된 응답 $y_c$ 의 보상이 비선호 $y_r$ 보다 크도록 학습합니다.

$$\mathcal{L}_{RM} = -\mathbb{E}\big[\log \sigma\big(r_\theta(x, y_c) - r_\theta(x, y_r)\big)\big]$$

**Step 3 — PPO 로 정책 최적화**: 강의자료의 최적화 목표

$$\mathbb{E}_{(x,y)\sim\pi^{RL}_\phi}\Big[r_\theta(x,y) - \beta\log\frac{\pi^{RL}_\phi(y|x)}{\pi^{SFT}(y|x)}\Big]
+ \lambda\,\mathbb{E}_{x\sim\pi_{\text{pretrain}}}\big[\log\pi^{RL}_\phi(x)\big]$$

* 두 번째 항($\beta$): SFT 모델에서 너무 멀어지지 않게 하는 **KL 페널티** (보상 해킹 방지)
* 세 번째 항($\lambda$): 사전학습 성능이 무너지는 것을 막는 **pretraining mix**

## 할 일
1. `reward_ranking_loss(r_chosen, r_rejected)` — Step 2 손실 (`logsigmoid` 사용)
2. `sequence_logprob(logits, tokens)` — 시퀀스별 로그 확률 $\log \pi(y|x) = \sum_t \log \pi(y_t|\cdot)$
3. `instructgpt_objective(reward, logp_rl, logp_sft, pretrain_logp, beta, lam)` — 위 식 (**최대화** 대상)

In [14]:
def reward_ranking_loss(r_chosen, r_rejected):
    """선호 응답의 보상이 비선호 응답보다 크도록 하는 손실."""
    return float(-np.mean(logsigmoid(np.asarray(r_chosen) - np.asarray(r_rejected))))


def sequence_logprob(logits, tokens):
    """시퀀스별 로그 확률의 시간축 합."""
    logp = log_softmax(logits, axis=-1)
    tokens = np.asarray(tokens)
    chosen = np.take_along_axis(logp, tokens[..., None], axis=-1)[..., 0]
    return chosen.sum(axis=1)


def instructgpt_objective(reward, logp_rl, logp_sft, pretrain_logp, beta=0.02, lam=0.1):
    """최대화 대상 objective와 평균 KL 추정값."""
    reward = np.asarray(reward)
    logp_rl = np.asarray(logp_rl)
    logp_sft = np.asarray(logp_sft)
    pretrain_logp = np.asarray(pretrain_logp)
    kl = logp_rl - logp_sft
    objective = np.mean(reward - beta * kl) + lam * np.mean(pretrain_logp)
    return float(objective), float(np.mean(kl))


In [15]:
# ===================== 문제 7 채점 =====================
rng = np.random.default_rng(77)

# --- Step 2: 보상 모델 ---
rc = np.array([2.0, 1.0, 0.5])
rr = np.array([1.0, 1.0, 3.0])
got = reward_ranking_loss(rc, rr)
ref = float(np.mean([-np.log(1 / (1 + np.exp(-(a - b)))) for a, b in zip(rc, rr)]))
assert abs(got - ref) < 1e-9, f"reward_ranking_loss 불일치: {got} vs {ref}"
ok(f"reward_ranking_loss = {got:.6f}")

tie = reward_ranking_loss(np.array([1.0]), np.array([1.0]))
assert abs(tie - np.log(2)) < 1e-9, "보상이 같으면 손실이 log2 여야 합니다"
ok(f"동점일 때 손실 = log2 = {tie:.6f}")

good = reward_ranking_loss(np.array([5.0]), np.array([-5.0]))
bad = reward_ranking_loss(np.array([-5.0]), np.array([5.0]))
assert good < 0.01 < bad, "선호/비선호 순서가 손실에 반영되지 않습니다"
ok(f"올바른 순위 {good:.5f} << 뒤집힌 순위 {bad:.5f}")

big = reward_ranking_loss(np.array([-800.0]), np.array([800.0]))
assert np.isfinite(big), "큰 값에서 오버플로가 납니다 (logsigmoid 를 쓰세요)"
ok("수치적으로 안정 (|Δr| = 1600 에서도 유한)")

# --- 시퀀스 로그 확률 ---
Bn, T, Vn = 3, 5, 11
lg = rng.normal(size=(Bn, T, Vn))
tk = rng.integers(0, Vn, size=(Bn, T))
sp = sequence_logprob(lg, tk)
assert sp.shape == (Bn,), "출력은 (B,) 이어야 합니다"
ref_sp = np.array([sum(log_softmax(lg[b, t])[tk[b, t]] for t in range(T)) for b in range(Bn)])
assert np.allclose(sp, ref_sp), "시퀀스 로그 확률이 다릅니다 (합인지 확인)"
assert np.all(sp < 0), "로그 확률은 음수여야 합니다"
ok(f"sequence_logprob 정상 (예: {sp[0]:.4f})")

# --- Step 3: PPO 목적함수 ---
N, M = 100, 50
reward = rng.normal(3.0, 1.0, size=N)
logp_sft = -rng.uniform(20, 30, size=N)
logp_rl = logp_sft + rng.normal(0.5, 0.3, size=N)
pre = -rng.uniform(10, 15, size=M)

obj, mkl = instructgpt_objective(reward, logp_rl, logp_sft, pre, beta=0.02, lam=0.1)
ref_kl = float((logp_rl - logp_sft).mean())
ref_obj = float((reward - 0.02 * (logp_rl - logp_sft)).mean() + 0.1 * pre.mean())
assert abs(mkl - ref_kl) < 1e-9, "KL 추정값이 틀립니다"
assert abs(obj - ref_obj) < 1e-9, f"목적함수 불일치: {obj} vs {ref_obj}"
ok(f"목적함수 = {obj:.6f}, 평균 KL = {mkl:.6f}")

# beta 를 키우면 (KL>0 인 상황에서) 목적함수 값이 줄어야 한다 = 페널티가 작동
o_lo, _ = instructgpt_objective(reward, logp_rl, logp_sft, pre, beta=0.0, lam=0.1)
o_hi, _ = instructgpt_objective(reward, logp_rl, logp_sft, pre, beta=1.0, lam=0.1)
assert o_hi < o_lo, "beta 를 키웠는데 KL 페널티가 작동하지 않습니다"
ok(f"KL 페널티 작동: beta=0 → {o_lo:.4f}, beta=1 → {o_hi:.4f}")

# SFT 와 동일한 정책이면 KL = 0
o_same, kl0 = instructgpt_objective(reward, logp_sft, logp_sft, pre, beta=5.0, lam=0.0)
assert abs(kl0) < 1e-12 and abs(o_same - reward.mean()) < 1e-9, "정책이 같으면 KL 항이 0이어야 합니다"
ok("π_RL = π_SFT 일 때 KL = 0")

# lam 을 키우면 pretraining mix 항이 반영된다 (pre < 0 이므로 목적함수 감소)
o_l0, _ = instructgpt_objective(reward, logp_rl, logp_sft, pre, beta=0.02, lam=0.0)
o_l1, _ = instructgpt_objective(reward, logp_rl, logp_sft, pre, beta=0.02, lam=1.0)
assert abs((o_l1 - o_l0) - pre.mean()) < 1e-9, "pretraining mix 항 반영이 틀립니다"
ok("pretraining mix(λ) 항 정상 반영")

print("\n✅ 문제 7 통과")

  [PASS] reward_ranking_loss = 1.195100
  [PASS] 동점일 때 손실 = log2 = 0.693147
  [PASS] 올바른 순위 0.00005 << 뒤집힌 순위 10.00005
  [PASS] 수치적으로 안정 (|Δr| = 1600 에서도 유한)
  [PASS] sequence_logprob 정상 (예: -13.5998)
  [PASS] 목적함수 = 1.727188, 평균 KL = 0.546104
  [PASS] KL 페널티 작동: beta=0 → 1.7381, beta=1 → 1.1920
  [PASS] π_RL = π_SFT 일 때 KL = 0
  [PASS] pretraining mix(λ) 항 정상 반영

✅ 문제 7 통과


> **PyTorch/실무 대응**: `trl` 라이브러리의 `RewardTrainer`(Step 2), `PPOTrainer`(Step 3).
> Ouyang et al., *Training Language Models to Follow Instructions with Human Feedback*, NeurIPS 2022.

---
# 마무리 — 전체 채점

아래 셀을 실행하면 7개 문제를 한 번에 다시 검증합니다.

In [16]:
# ===================== 전체 요약 채점 =====================
rng = np.random.default_rng(2025)
score, log = 0, []

def check(name, fn):
    global score
    try:
        fn(); score_add = 1; msg = "통과"
    except Exception as e:
        score_add = 0; msg = f"실패 ({type(e).__name__}: {e})"
    globals()['score'] = globals()['score'] + score_add
    log.append(f"  문제 {name:<40s} {msg}")

def _q1():
    ii = rng.integers(10, 100, size=(32, 16)); sp = np.zeros((32, 16), int); sp[:, 0] = 1
    m, l = mlm_mask_tokens(ii, 100, 4, sp, 0.15, np.random.default_rng(1))
    assert np.all(l[sp.astype(bool)] == -100) and np.all(m[l == -100] == ii[l == -100])
    lo = rng.normal(size=(2, 3, 5)); la = np.array([[1, -100, 2], [-100, 0, 4]])
    r = float(np.mean([-log_softmax(lo[b, t])[la[b, t]] for b in range(2) for t in range(3) if la[b, t] != -100]))
    assert abs(masked_cross_entropy(lo, la) - r) < 1e-9

def _q2():
    im = rng.normal(size=(2, 3, 8, 8)); x = patchify(im, 4)
    assert np.allclose(unpatchify(x, 4, 3, 8, 8), im)
    xk, mk, ir = random_masking(x, 0.75, np.random.default_rng(3))
    assert mk.sum() == 2 * 3 and xk.shape[1] == 1
    p, t = rng.normal(size=(1, 2, 3)), rng.normal(size=(1, 2, 3)); mm = np.array([[1., 0.]])
    assert abs(mae_loss(p, t, mm) - ((p[0, 0] - t[0, 0]) ** 2).mean()) < 1e-9

def _q3():
    a, b = rng.normal(size=(4, 6)), rng.normal(size=(4, 6))
    z = l2_normalize(np.concatenate([a, b], 0)); s = z @ z.T / 0.3
    np.fill_diagonal(s, -np.inf)
    tg = np.concatenate([np.arange(4, 8), np.arange(0, 4)])
    assert abs(nt_xent_loss(a, b, 0.3) + log_softmax(s, -1)[np.arange(8), tg].mean()) < 1e-9

def _q4():
    a, b = rng.normal(size=(5, 7)), rng.normal(size=(5, 7))
    lg = clip_logits(a, b, 10.0); lab = np.arange(5)
    r = float((-log_softmax(lg, -1)[lab, lab].mean() - log_softmax(lg.T, -1)[lab, lab].mean()) / 2)
    assert abs(clip_loss(a, b, 10.0) - r) < 1e-9
    p = l2_normalize(rng.normal(size=(3, 7)))
    assert np.array_equal(zero_shot_classify(p * 2.0, p), np.arange(3))

def _q5():
    q, k, v = (rng.normal(size=(2, 4, 6)) for _ in range(3))
    o, at = causal_self_attention(q, k, v)
    assert np.allclose(at[:, np.triu(np.ones((4, 4), bool), 1)], 0) and np.allclose(o[:, 0], v[:, 0])
    lo = rng.normal(size=(2, 4, 6)); tk = rng.integers(0, 6, size=(2, 4))
    r = float(np.mean([-log_softmax(lo[b, t])[tk[b, t + 1]] for b in range(2) for t in range(3)]))
    assert abs(lm_loss_L1(lo, tk) - r) < 1e-9
    cl = rng.normal(size=(2, 3)); y = np.array([0, 2])
    tot, cc, cm = gpt1_objective(cl, y, lo, tk, 0.5)
    assert abs(tot - (cc + 0.5 * cm)) < 1e-9

def _q6():
    X, Y, W = rng.normal(size=(24, 4)), rng.normal(size=(24, 2)), rng.normal(size=(4, 2))
    assert np.allclose(data_parallel_grad(X, Y, W, 6), mse_grad(X, Y, W), atol=1e-12)
    A, Bm = rng.normal(size=(6, 12)) * .3, rng.normal(size=(12, 5)) * .3
    Xt = rng.normal(size=(7, 6))
    assert np.allclose(tensor_parallel_mlp(Xt, A, Bm, 4), mlp_single_device(Xt, A, Bm), atol=1e-10)

def _q7():
    assert abs(reward_ranking_loss(np.array([0.]), np.array([0.])) - np.log(2)) < 1e-9
    lo = rng.normal(size=(2, 3, 5)); tk = rng.integers(0, 5, size=(2, 3))
    r = np.array([sum(log_softmax(lo[b, t])[tk[b, t]] for t in range(3)) for b in range(2)])
    assert np.allclose(sequence_logprob(lo, tk), r)
    rw, pl, sl, pr = rng.normal(size=8), rng.normal(size=8), rng.normal(size=8), rng.normal(size=4)
    o, kl = instructgpt_objective(rw, pl, sl, pr, 0.02, 0.1)
    assert abs(o - ((rw - 0.02 * (pl - sl)).mean() + 0.1 * pr.mean())) < 1e-9

score = 0
for nm, f in [("1  MLM (생성적 사전학습)", _q1), ("2  MAE (마스크 이미지 모델링)", _q2),
              ("3  SimCLR NT-Xent", _q3), ("4  CLIP & 제로샷", _q4),
              ("5  GPT-1 인과 어텐션/목적함수", _q5), ("6  데이터·텐서 병렬", _q6),
              ("7  InstructGPT RLHF", _q7)]:
    check(nm, f)

print("=" * 62)
print("\n".join(log))
print("=" * 62)
print(f"최종 점수: {score} / 7")
if score == 7:
    print("🎉 전부 통과했습니다. 수고하셨습니다!")
else:
    print("아직 통과하지 못한 문제가 있습니다. 위 로그를 확인하세요.")

  문제 1  MLM (생성적 사전학습)                        통과
  문제 2  MAE (마스크 이미지 모델링)                     통과
  문제 3  SimCLR NT-Xent                        통과
  문제 4  CLIP & 제로샷                            통과
  문제 5  GPT-1 인과 어텐션/목적함수                     통과
  문제 6  데이터·텐서 병렬                             통과
  문제 7  InstructGPT RLHF                      통과
최종 점수: 7 / 7
🎉 전부 통과했습니다. 수고하셨습니다!


---
## 정리 — 오늘 확인한 것

| 개념 | 핵심 한 줄 |
|---|---|
| 생성적 사전학습 | 입력을 일부러 망가뜨리고 **원본을 복원**시키면 라벨 없이 표현이 학습된다 (MLM=텍스트, MAE=이미지) |
| 대조적 사전학습 | 양성 쌍은 당기고 음성 쌍은 밀어낸다. **손실 하나로** 표현 공간의 기하가 결정된다 |
| 멀티모달 대조학습 | 이미지·텍스트를 같은 공간에 정렬하면 **라벨 없이도 분류기**가 생긴다 (CLIP 제로샷) |
| GPT 목적함수 | 인과 마스크가 $L_1$ 을 성립시키고, $L_3=L_2+\lambda L_1$ 로 사전학습 지식을 유지한 채 미세조정한다 |
| 병렬 학습 | 데이터 병렬은 **기울기 평균**, 텐서 병렬은 **열/행 분할 + all-reduce**. 둘 다 단일 장치와 수학적으로 동일 |
| RLHF | 보상 최대화만 하면 모델이 망가지므로 **KL 페널티**로 SFT 근처에 묶어 둔다 |

## 더 해 볼 것 (선택)
1. 문제 3의 `tau` 를 0.01 ~ 5.0 으로 바꿔가며 손실을 그려 보고, 온도가 "음성 샘플에 대한 민감도"를 어떻게 조절하는지 설명해 보세요.
2. 문제 4에서 이미지 임베딩의 노이즈 크기를 키우며 제로샷 정확도가 어떻게 무너지는지 곡선을 그려 보세요.
3. 문제 6의 텐서 병렬에서 all-reduce 통신량(원소 개수)을 장치 수의 함수로 계산해 보고, 왜 hidden 차원이 커질수록 텐서 병렬이 유리한지 논하세요.
4. 문제 7에서 보상은 높지만 KL 도 큰 "보상 해킹" 샘플을 인위적으로 만들어, `beta` 를 얼마나 키워야 그 샘플이 억제되는지 찾아보세요.